# EXAFS over a whole run — escape + dask

The single-spectrum functions in `escape.exafs` reduce one `μ(E)` curve. At an
FEL or a scanning beamline you have **many** spectra stacked along the event
axis of an `escape.Array` (one per pulse, scan step, or repeat). This notebook
shows the batch layer — `reduce_array` and `ft_array` — which reduces the whole
stack **lazily and in parallel** over dask, exactly like
`escape.wavefront.propagate_array`.

We fake a run by adding noise to the bundled Cu-foil spectrum.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import dask.array as da

import escape
from escape import exafs
import escape.exafs

plt.rcParams['figure.figsize'] = (6, 4)

## 1. A reference spectrum fixes E0 and the edge step

In [ ]:
data = os.path.join(os.path.dirname(escape.exafs.__file__), "data", "cu_rt01.xmu")
energy, mu = exafs.read_columns(data)
pre = exafs.pre_edge(energy, mu)
print(f"reference  E0 = {pre.e0:.2f} eV   edge step = {pre.edge_step:.3f}")

## 2. Build a stack of spectra as an escape Array

Shape is `(n_events, n_energy)` — the event axis first, the shared `energy` axis
second. In a real analysis this Array comes straight from your parser.

In [ ]:
rng = np.random.default_rng(0)
n_events = 24
stack = np.array([mu + rng.normal(0, 0.004, mu.size) for _ in range(n_events)])

mu_arr = escape.Array(data=da.from_array(stack, chunks=(6, mu.size)),
                      index=np.arange(n_events))
print("mu Array:", mu_arr.data.shape, "chunks:", mu_arr.data.chunks[0],
      "lazy:", mu_arr.is_dask_array())

## 3. Reduce the stack to χ(k), lazily

We pass fixed `e0`/`edge_step` from the reference so every event lands on the
same scale. `reduce_array` returns a lazy Array `(n_events, n_k)`.

In [ ]:
kgrid = exafs.common_k_grid(0, 16, 0.05)
chi = exafs.reduce_array(mu_arr, energy, kgrid,
                         e0=pre.e0, edge_step=pre.edge_step, rbkg=1.0)
print("chi(k) Array:", chi.data.shape, "lazy:", chi.is_dask_array())

# nothing computed yet; trigger it now and average over events
chik = chi.data.compute()
plt.plot(kgrid, kgrid**2 * chik.T, color='0.7', lw=0.4)
plt.plot(kgrid, kgrid**2 * chik.mean(0), 'C0', lw=1.6, label='mean over run')
plt.axhline(0, color='k', lw=0.5); plt.xlabel(r'$k$ (Å$^{-1}$)')
plt.ylabel(r'$k^2\chi(k)$'); plt.legend(); plt.title('per-event and mean EXAFS'); plt.show()

## 4. Fourier transform the stack to χ(R), lazily

In [ ]:
r, chir = exafs.ft_array(chi, kgrid, kweight=2, window='hanning', kmin=3, kmax=13, dk=1.0)
print("chi(R) Array:", chir.data.shape, chir.data.dtype)

chir_mag = np.abs(chir.data).compute()
plt.plot(r, chir_mag.T, color='0.7', lw=0.4)
plt.plot(r, chir_mag.mean(0), 'C3', lw=1.8, label='mean |χ(R)|')
peak = r[(r>1)&(r<3)][np.argmax(chir_mag.mean(0)[(r>1)&(r<3)])]
plt.axvline(peak, color='C3', ls=':', lw=1)
plt.xlim(0, 6); plt.xlabel(r'$R$ (Å, not phase-corrected)'); plt.ylabel(r'$|\chi(R)|$')
plt.legend(); plt.title(f'first-shell peak at R = {peak:.2f} Å'); plt.show()
print('Cu–Cu is 2.55 Å; the ~0.3 Å shortfall is the EXAFS phase shift.')

## 5. It composes with the rest of escape

Because `chi` and `chir` are ordinary lazy `escape.Array`s sharing the event
index, they slot into the usual escape machinery — group by a scan parameter,
filter on a condition, average — all before `.compute()`. For example the
event-averaged χ(k) is just:

In [ ]:
mean_chik = chi.mean(axis=0)            # still lazy
print(type(mean_chik).__name__, '->', mean_chik.data.shape, '  compute mean rms:',
      round(float(np.sqrt((mean_chik.data.compute()**2).mean())), 4))

## References

* AUTOBK background: Newville, Līviņš, Yacoby, Rehr & Stern, *Phys. Rev. B* **47**, 14126 (1993). doi:[10.1103/PhysRevB.47.14126](https://doi.org/10.1103/PhysRevB.47.14126)
* Fourier-transform EXAFS: Sayers, Stern & Lytle, *Phys. Rev. Lett.* **27**, 1204 (1971). doi:[10.1103/PhysRevLett.27.1204](https://doi.org/10.1103/PhysRevLett.27.1204)
* Conventions / `ETOK` from IFEFFIT/Larch: Newville, *J. Synchrotron Rad.* **8**, 322 (2001); <https://xraypy.github.io/xraylarch/>
* Demo data: Cu foil, APS 13ID, from [xraylarch](https://github.com/xraypy/xraylarch) examples.
* Production analysis / shell fitting: [Larch](https://xraypy.github.io/xraylarch/), [Demeter/Artemis](https://bruceravel.github.io/demeter/).